## Part 4: Models, metrics and XAI (P4)

This part trains and compares the models on the same modeling table, chooses the metrics and the threshold without touching the test set, and explains the models.

Sections: 4.1) shared evaluation function `train_eval()`, 4.2) draft data, 4.3) sanity check of the metrics, 4.4) baselines, 4.5) statistical models.

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import load_clean, SEED

In [ ]:
clean = load_clean()
base = clean["model_base"]      # pre-race columns + the target `podium`
outcomes = clean["outcomes"]    # post-race columns (status, points...). Never features
tables = clean["tables"]        # all other cleaned tables

### 4.1 The evaluation function `train_eval()`

One function scores every model the same way, so all results (our models and P3's ablation) are directly comparable.

**Why these metrics?** The brief asks for ROC-AUC, PR-AUC and F1 for every model; we also report precision and recall, because F1 alone hides which of the two is weak.
* **ROC-AUC:** how well the model separates podium from non-podium overall (uses probabilities, no threshold).
* **PR-AUC:** how well the model ranks the real podium finishers at the top (uses probabilities). Computed as *average precision*, the standard summary of the precision-recall curve. It is more informative than ROC-AUC when the positive class is rare, and podiums are rare (about 14% of rows).
* **F1, precision, recall:** quality of the yes/no decision at a chosen probability threshold. The threshold is 0.5 for now; it will be tuned on validation in a later step.

**Leakage safety:** the model is fitted on the training years only. The test set is deliberately *not* an input: it is scored once, at the very end, with `evaluate()`.

The same code is in `src/f1_eval.py`, so P3 can import it for the ablation.

In [ ]:
import pandas as pd
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


def evaluate(model, X, y, threshold=0.5):
    """Score an already-fitted model on one split and return the 5 metrics.
    ROC-AUC and PR-AUC use the probabilities (no threshold needed);
    F1, precision and recall use the yes/no prediction at `threshold`."""
    proba = model.predict_proba(X)[:, 1]          # predicted probability of a podium
    pred = (proba >= threshold).astype(int)       # 1 = "podium" at this threshold
    return {
        "ROC-AUC": roc_auc_score(y, proba),
        "PR-AUC": average_precision_score(y, proba),
        "F1": f1_score(y, pred, zero_division=0),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
    }


def train_eval(model, X_train, y_train, X_val, y_val, threshold=0.5, name="model"):
    """Fit `model` on the training years only, then score it on train and validation.
    The test set is not used here on purpose: it is scored once, at the end, with evaluate()."""
    model.fit(X_train, y_train)
    rows = [
        {"model": name, "split": "train", **evaluate(model, X_train, y_train, threshold)},
        {"model": name, "split": "validation", **evaluate(model, X_val, y_val, threshold)},
    ]
    return pd.DataFrame(rows).round(4)

### 4.2 Draft data (temporary)

P3's modeling table is not ready yet (draft on Oct 8, frozen on Oct 10). To test the function now, we use `model_base` with the team's time split (train <= 2019, validation 2020-2021) and four pre-race columns. This cell will be replaced by P3's table; the test years (>= 2022) are not touched here.

In [ ]:
DRAFT_FEATURES = ["grid", "pitlane_start", "has_qualifying", "qual_position"]

train = base[base["year"] <= 2019]
val = base[base["year"].between(2020, 2021)]

X_train, y_train = train[DRAFT_FEATURES].astype(float), train["podium"]
X_val, y_val = val[DRAFT_FEATURES].astype(float), val["podium"]

print(f"train: {len(X_train)} rows, podium rate {y_train.mean():.3f}")
print(f"validation: {len(X_val)} rows, podium rate {y_val.mean():.3f}")

### 4.3 Sanity check of the metrics

Before trusting the function, we test it on a model that knows nothing: `DummyClassifier` always predicts the training podium rate. A correct implementation must give **ROC-AUC = 0.5** (no separation) and **PR-AUC = the podium rate** (no better than random ranking), and **F1 = 0** at threshold 0.5 (it never says "podium"). We also print its accuracy, to show why accuracy is not one of our metrics.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="prior", random_state=SEED)
dummy_results = train_eval(dummy, X_train, y_train, X_val, y_val, name="dummy (always the base rate)")
display(dummy_results)

val_row = dummy_results[dummy_results["split"] == "validation"].iloc[0]
assert abs(val_row["ROC-AUC"] - 0.5) < 1e-3
assert abs(val_row["PR-AUC"] - y_val.mean()) < 1e-3
assert val_row["F1"] == 0
print(f"Dummy accuracy on validation: {accuracy_score(y_val, dummy.predict(X_val)):.1%} "
      f"(looks good, but it finds 0 podiums)")

**What we see.** The dummy model gets ROC-AUC 0.5, PR-AUC equal to the podium rate and F1 = 0, exactly as expected, so the function computes the metrics correctly. Its accuracy is still about 85%, although it never finds a single podium. That is why we do not use accuracy, and why **PR-AUC is the natural primary metric** for this rare positive class. These dummy numbers are also the floor every real model must beat.

**Next step:** the baselines in section 4.4.

### 4.4 Baselines: a `grid <= 3` rule and logistic regression

A baseline is the simplest reasonable approach. Every later model must beat it, otherwise its extra complexity is not earned. We use two:
* **Rule `grid <= 3`:** predict a podium for the three cars that start at the front. It learns nothing; it only encodes the domain fact that the starting position is the strongest single predictor of a podium.
* **Logistic regression:** the simplest statistical model. It multiplies each feature by a learned weight, adds them up, and turns the sum into a probability with the sigmoid function. *Limitation:* it is linear, so it cannot learn interactions on its own (for example "a good grid slot matters more in a reliable era").

Same draft data as section 4.2; the test years are not used.

#### Attempt 1: rule `grid <= 3`
The rule gives a hard yes/no, so its ROC-AUC and PR-AUC are those of a yes/no score (a single operating point). `grid = 0` (pit-lane start, no grid slot) is not counted as a front-row start.

In [ ]:
import numpy as np

class GridRule:
    """Rule baseline: 'podium' if the car starts in grid slot 1, 2 or 3. Learns nothing."""
    def fit(self, X, y):
        return self

    def predict_proba(self, X):
        p = X["grid"].between(1, 3).astype(float).to_numpy()
        return np.column_stack([1 - p, p])

rule_results = train_eval(GridRule(), X_train, y_train, X_val, y_val, name="rule grid <= 3")
display(rule_results)

#### Attempt 2: logistic regression
`qual_position` is missing for most rows before the mid-1990s (no qualifying data, see Part 1), so the model needs an imputer. **Temporary preprocessing** until P3's pipeline is ready: fill missing values with the training median (`has_qualifying` already tells the model the value was missing), then standardize, so the weights are comparable and the solver converges. Both steps sit inside a `Pipeline`, so `train_eval()` fits them on the training years only.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                   LogisticRegression(max_iter=1000, random_state=SEED))
lr_results = train_eval(lr, X_train, y_train, X_val, y_val, name="logistic regression")
display(lr_results)

#### Attempt 3: logistic regression with `class_weight="balanced"`
Podiums are only about 14% of the training rows. `class_weight="balanced"` gives the podium rows a larger weight, so both classes count equally in training. We expect the ranking metrics (ROC-AUC, PR-AUC) to stay about the same and the yes/no metrics at threshold 0.5 to change, because the weighting mostly moves the predicted probabilities up.

In [ ]:
lr_bal = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))
lr_bal_results = train_eval(lr_bal, X_train, y_train, X_val, y_val, name="logistic regression (balanced)")
display(lr_bal_results)

#### Comparison of the baselines (validation)

In [ ]:
import matplotlib.pyplot as plt

baselines = pd.concat([dummy_results, rule_results, lr_results, lr_bal_results], ignore_index=True)
val_table = baselines[baselines["split"] == "validation"].set_index("model").drop(columns="split")
display(val_table)

ax = val_table[["ROC-AUC", "PR-AUC", "F1"]].plot.barh(figsize=(8, 4))
ax.set_title("Baselines on validation (2020-2021), threshold 0.5")
ax.set_xlabel("score (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(title="metric", loc="lower right")
plt.tight_layout()
plt.show()

# what the logistic regression learned (standardized features, so the sizes are comparable)
coefs = pd.Series(lr.named_steps["logisticregression"].coef_[0], index=DRAFT_FEATURES)
display(coefs.sort_values().round(3).to_frame("weight (standardized)"))

**What we see (validation 2020-2021).**
* **Both baselines are far above the dummy** (PR-AUC 0.15), so the draft features carry real signal.
* **Logistic regression is the better ranker:** PR-AUC 0.667 vs 0.530 for the rule, and ROC-AUC 0.916 vs 0.820. On our primary metric (PR-AUC) it is the baseline to beat.
* **At threshold 0.5 the rule has the best F1** (0.695 vs 0.591). Plain logistic regression is too cautious at 0.5: precision 0.75 but recall only 0.49. This shows the threshold matters as much as the model, so it will be tuned on validation in a later step instead of staying at 0.5.
* **`class_weight="balanced"` behaved as expected:** PR-AUC and ROC-AUC barely change (0.666 vs 0.667), because the ranking stays the same; at 0.5 recall jumps to 0.91 and precision drops to 0.40. Weighting mostly moves the probabilities, which threshold tuning can also do.
* **Weights:** `grid` has by far the largest weight (-2.05: a lower grid number, i.e. a starting slot nearer the front, means a higher podium probability), then `qual_position` (-0.49). These are associations the model uses, not causes; `has_qualifying` and `pitlane_start` are also correlated with the era and with `grid`, so their individual signs should not be over-read.
* **Train scores are lower than validation** (PR-AUC 0.53 vs 0.67). This is not underfitting: the training years go back to 1950, when big fields and low reliability made the grid a weaker predictor. Era sensitivity is a known limitation, and the training start year is a decision for P3's split.

**Baseline to beat:** logistic regression, validation PR-AUC **0.667**. **Next step:** random forest and gradient boosting in section 4.5.


### 4.5 Statistical models: random forest and gradient boosting

Two tree-based models, chosen because they can learn what logistic regression cannot: non-linear effects and interactions between features. Both are in scikit-learn, so no new library is needed.
* **Random forest:** many decision trees, each trained on a random sample of rows and features; the prediction is the average vote. Averaging many different trees reduces overfitting. *Limitation:* large and slow-ish, and its probabilities are averages of votes, not calibrated.
* **Gradient boosting** (`HistGradientBoostingClassifier`): small trees built one after another, each one correcting the errors of the trees before it. Usually the strongest model on tables like ours. *Limitation:* more hyperparameters, and it can overfit if it runs too long (scikit-learn stops early on a 10% slice of the training rows, never on our validation years).

**Class imbalance:** both use `class_weight="balanced"`, as required for the statistical models (podiums are about 14% of rows). Section 4.4 showed this mostly moves the probabilities; the threshold is tuned later.

Same draft data and temporary preprocessing as section 4.4; hyperparameters are tuned in a later step.

#### Attempt 4: random forest
300 trees; `min_samples_leaf=20` so every leaf is based on at least 20 races, which keeps the probabilities smooth instead of memorizing single rows. The median imputer is kept so the model gets exactly the same inputs as logistic regression.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = make_pipeline(SimpleImputer(strategy="median"),
                   RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced",
                                          n_jobs=-1, random_state=SEED))
rf_results = train_eval(rf, X_train, y_train, X_val, y_val, name="random forest")
display(rf_results)

#### Attempt 5: gradient boosting
`HistGradientBoostingClassifier` handles missing values natively (it learns which side of each split the missing rows go to), so it needs no imputer and no scaling. Default settings for now.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED)
gb_results = train_eval(gb, X_train, y_train, X_val, y_val, name="gradient boosting")
display(gb_results)

#### Comparison of all models so far
Validation scores for every attempt, plus the train-validation gap in PR-AUC (a large positive gap would mean overfitting).

In [ ]:
all_models = pd.concat([dummy_results, rule_results, lr_results, lr_bal_results, rf_results, gb_results],
                       ignore_index=True)
pr = all_models.pivot(index="model", columns="split", values="PR-AUC")
summary = all_models[all_models["split"] == "validation"].set_index("model").drop(columns="split")
summary["PR-AUC gap (train - val)"] = (pr["train"] - pr["validation"]).round(4)
display(summary.sort_values("PR-AUC", ascending=False))

lr_pr = summary.loc["logistic regression", "PR-AUC"]
ax = summary["PR-AUC"].sort_values().plot.barh(figsize=(8, 4), color="tab:orange")
ax.axvline(lr_pr, color="black", linestyle="--", label=f"baseline to beat (logistic regression) = {lr_pr:.4f}")
ax.set_title("PR-AUC on validation (2020-2021), all models so far")
ax.set_xlabel("PR-AUC (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**What we see (validation 2020-2021).**
* **The tree models do not beat logistic regression on the draft data.** PR-AUC: gradient boosting 0.6674, logistic regression 0.6665, random forest 0.6661; ROC-AUC is 0.915-0.917 for all three. Differences in the third decimal, with only about 117 podiums in the validation years, are a tie.
* **Why:** the draft table has only four features, and one of them, `grid`, carries almost all the signal in a simple shape (the nearer the front, the likelier the podium). A straight line already captures that, so the strengths of trees (non-linear effects, interactions) have nothing extra to learn. This is the brief's point that feature engineering matters as much as model choice: the comparison has to be repeated on P3's table, where form and standings features give the trees something to work with.
* **No overfitting:** the train-validation PR-AUC gap is about -0.14 for every model, the same era effect seen in section 4.4 (the training years go back to 1950). The forest (`min_samples_leaf=20`) and boosting (early stopping on training rows) did not memorize the training data.
* **At threshold 0.5** both balanced tree models behave like the balanced logistic regression: high recall (0.86-0.88), low precision (0.43-0.45). The threshold is tuned on validation in a later step.

**Decision:** keep random forest and gradient boosting as our two statistical models and re-run them on P3's table. The baseline to beat is still logistic regression. **Next step:** the shallow neural network.
